## Extract Poses from Amass Dataset

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook
%matplotlib inline

import sys, os
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from tqdm import tqdm



from human_body_prior.tools.omni_tools import copy2cpu as c2c

os.environ['PYOPENGL_PLATFORM'] = 'egl'

### Please remember to download the following subdataset from AMASS website: https://amass.is.tue.mpg.de/download.php. Note only download the <u>SMPL+H G</u> data.
* ACCD (ACCD)
* HDM05 (MPI_HDM05)
* TCDHands (TCD_handMocap)
* SFU (SFU)
* BMLmovi (BMLmovi)
* CMU (CMU)
* Mosh (MPI_mosh)
* EKUT (EKUT)
* KIT  (KIT)
* Eyes_Janpan_Dataset (Eyes_Janpan_Dataset)
* BMLhandball (BMLhandball)
* Transitions (Transitions_mocap)
* PosePrior (MPI_Limits)
* HumanEva (HumanEva)
* SSM (SSM_synced)
* DFaust (DFaust_67)
* TotalCapture (TotalCapture)
* BMLrub (BioMotionLab_NTroje)

### Unzip all datasets. In the bracket we give the name of the unzipped file folder. Please correct yours to the given names if they are not the same.

### Place all files under the directory **./amass_data/**. The directory structure shoud look like the following:  
./amass_data/  
./amass_data/ACCAD/  
./amass_data/BioMotionLab_NTroje/  
./amass_data/BMLhandball/  
./amass_data/BMLmovi/   
./amass_data/CMU/  
./amass_data/DFaust_67/  
./amass_data/EKUT/  
./amass_data/Eyes_Japan_Dataset/  
./amass_data/HumanEva/  
./amass_data/KIT/  
./amass_data/MPI_HDM05/  
./amass_data/MPI_Limits/  
./amass_data/MPI_mosh/  
./amass_data/SFU/  
./amass_data/SSM_synced/  
./amass_data/TCD_handMocap/  
./amass_data/TotalCapture/  
./amass_data/Transitions_mocap/  

**Please make sure the file path are correct, otherwise it can not succeed.**

In [ ]:
# Choose the device to run the body model on.
comp_device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [ ]:
from human_body_prior.body_model.body_model import BodyModel

male_bm_path = './body_models/smplh/male/model.npz'
male_dmpl_path = './body_models/dmpls/male/model.npz'

female_bm_path = './body_models/smplh/female/model.npz'
female_dmpl_path = './body_models/dmpls/female/model.npz'

num_betas = 10 # number of body parameters
num_dmpls = 8 # number of DMPL parameters

male_bm = BodyModel(
    bm_fname=male_bm_path, num_betas=num_betas, num_dmpls=num_dmpls, dmpl_fname=male_dmpl_path
).to(comp_device)
faces = c2c(male_bm.f)

female_bm = BodyModel(
    bm_fname=female_bm_path, num_betas=num_betas, num_dmpls=num_dmpls, dmpl_fname=female_dmpl_path
).to(comp_device)

In [ ]:
paths = []
folders = []
dataset_names = []
for root, dirs, files in os.walk('./amass_data'):
#     print(root, dirs, files)
#     for folder in dirs:
#         folders.append(os.path.join(root, folder))
    folders.append(root)
    for name in files:
        dataset_name = root.split('/')[2]
        if dataset_name not in dataset_names:
            dataset_names.append(dataset_name)
        paths.append(os.path.join(root, name))

In [ ]:
save_root = './pose_data'
save_folders = [folder.replace('./amass_data', './pose_data') for folder in folders]
print(save_folders)
for folder in save_folders:
    os.makedirs(folder, exist_ok=True)
group_path = [[path for path in paths if name in path] for name in dataset_names]
print(group_path)

In [ ]:
import csv
BASE_PATH = "../video/annotation-script"

def read_from_csv(csv_path):
    csv_info = {}

    with open(csv_path) as csv_file:
        csv_iter =  csv.reader(csv_file, delimiter=",")
        
        for row in csv_iter:
            video, gender, _, _ = row
            video_name = os.path.basename(video).replace(".mp4", "")
            csv_info[video_name] = gender
            
    return csv_info

csv_paths = os.path.join(BASE_PATH, "video-csv")
csv_files = sorted(
    os.listdir(csv_paths)
)

content_per_csv = {}
for csv_file in csv_files: 
    print(f"Running for '{csv_file}' file")
    csv_path = os.path.join(csv_paths, csv_file)
    csv_rows = read_from_csv(csv_path)
    content_per_csv[csv_file] = csv_rows

In [ ]:
trans_matrix = np.array([[1.0, 0.0, 0.0],
                            [0.0, 0.0, 1.0],
                            [0.0, 1.0, 0.0]])


def is_valid_video(video) -> bool:
    black_list = [
        "C.-CARRIOLA-TERRA-3quarti-1",
        "A.-TERRA-CARRIOLA-3quarti-2",
        "D.-TERRA-CARRIOLA-3quarti-2",
        "D.-CARRIOLA-TERRA-3quarti-1",
        "D.-CARRIOLA-TERRA-3quarti-2",
        "D.-TERRA-CARRIOLA-3quarti-3",
        "A.-CARRIOLA-TERRA-3quarti-1",
        "C.-TERRA-CARRIOLA-3quarti-2",
        "D.-CARRIOLA-TERRA-3quarti-3",
        "D.-TERRA-CARRIOLA-3quarti-1",
        "A.-TERRA-CARRIOLA-3quarti-1",
        "A.-TERRA-CASSA-O--LAT-1"
    ]

    return video not in black_list

ex_fps = 20
def amass_to_pose(src_path, save_path):
    fps = 100

    with np.load(src_path, allow_pickle=True) as bdata:
        is_pose = bdata.get('trans') is None

        if is_pose:
            return fps

        down_sample = int(fps / ex_fps)
        bm = None

        if "Custom" in src_path:
            for v in content_per_csv.values():
                gender = v.get(name.replace(".npz", ""))

                if gender is None:
                    continue

                if gender == 'male':
                    bm = male_bm
                    break
                else:
                    bm = female_bm
                    break
        else:
            if bdata['gender'] == 'male':
                bm = male_bm
            else:
                bm = female_bm
                
        body_params = {}
        bdata_poses = bdata['poses'][::down_sample,...]
        bdata_trans = bdata['trans'][::down_sample,...]
        betas = None
        
        if "Custom" in src_path:
            betas = bdata['betas'][0]
            betas = np.repeat(
                betas[:num_betas][np.newaxis], repeats=len(bdata_trans), 
                axis=0
            )
        else:
            betas = np.repeat(
                bdata['betas'][:num_betas][np.newaxis], repeats=len(bdata_trans), 
                axis=0
            )

        body_params = {
                'root_orient': torch.Tensor(bdata_poses[:, :3]).to(comp_device),
                'pose_body': torch.Tensor(bdata_poses[:, 3:66]).to(comp_device),
                'pose_hand': torch.Tensor(bdata_poses[:, 66:]).to(comp_device),
                'trans': torch.Tensor(bdata_trans).to(comp_device),
                'betas': torch.Tensor(betas).to(comp_device),
            }
            
        with torch.no_grad():
            body = bm(**body_params)
        pose_seq_np = body.Jtr.detach().cpu().numpy()
        pose_seq_np_n = np.dot(pose_seq_np, trans_matrix)
        np.save(save_path, pose_seq_np_n)
        
        return fps

In [ ]:
group_path = group_path
all_count = sum([len(paths) for paths in group_path])
cur_count = 0

This will take a few hours for all datasets, here we take one dataset as an example

To accelerate the process, you could run multiple scripts like this at one time.

In [ ]:
import time
for paths in group_path:
    dataset_name = paths[0].split('/')[2]
    pbar = tqdm(paths)
    pbar.set_description('Processing: %s'%dataset_name)
    fps = 0
    for path in pbar:
        save_path = path.replace('./amass_data', './pose_data')
        save_path = save_path[:-3] + 'npy'
        try:

            if os.path.exists(save_path):
                print("File already processed: ", save_path)
                continue

            name = os.path.basename(path)
        
            if "Custom" in path:
                if not is_valid_video(name.replace(".npz", "")):
                    print("Ignoring erroneus video: ", path)
                    continue
            

            fps = amass_to_pose(path, save_path)
        except Exception as e:
            print("Original path:", path)
            print("Save path:", save_path)
            print(e)
        
    cur_count += len(paths)
    print('Processed / All (fps %d): %d/%d'% (fps, cur_count, all_count) )
    time.sleep(0.5)

    # Controlla qui perche' non vengono rielaborati

The above code will extract poses from **AMASS** dataset, and put them under directory **"./pose_data"**

The source data from **HumanAct12** is already included in **"./pose_data"** in this repository. You need to **unzip** it right in this folder.

## Segment, Mirror and Relocate Motions

In [ ]:
import codecs as cs
import pandas as pd
import numpy as np
from tqdm import tqdm
from os.path import join as pjoin

In [ ]:
def swap_left_right(data):
    assert len(data.shape) == 3 and data.shape[-1] == 3
    data = data.copy()
    data[..., 0] *= -1
    right_chain = [2, 5, 8, 11, 14, 17, 19, 21]
    left_chain = [1, 4, 7, 10, 13, 16, 18, 20]
    left_hand_chain = [22, 23, 24, 34, 35, 36, 25, 26, 27, 31, 32, 33, 28, 29, 30]
    right_hand_chain = [43, 44, 45, 46, 47, 48, 40, 41, 42, 37, 38, 39, 49, 50, 51]
    tmp = data[:, right_chain]
    data[:, right_chain] = data[:, left_chain]
    data[:, left_chain] = tmp
    if data.shape[1] > 24:
        tmp = data[:, right_hand_chain]
        data[:, right_hand_chain] = data[:, left_hand_chain]
        data[:, left_hand_chain] = tmp
    return data

In [ ]:
import csv

def save_custom_csv():
    pose_path = "./pose_data/Custom"
    npz_path = "./amass_data/Custom"
    files = sorted(os.listdir(pose_path))

    paths = [
        os.path.join(pose_path, f)
        for f in files
    ]


    if os.path.exists("custom.csv"):
        os.remove("custom.csv")

    with open("custom.csv", mode="w") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["source_path","start_frame","end_frame","new_name"])

        for pairs in zip(files, paths):
            file_name, path = pairs
            
            npz_file_name = file_name.replace(".npy", ".npz")
            with  np.load(f"{npz_path}/{npz_file_name}") as bdata:
                frames = bdata["poses"].shape[0] - 1
                row = [path, 0, frames, file_name]
                writer.writerow(row)
    

In [ ]:
def save_to_npz(csv_path):
    save_dir = './joints'
    index_file = pd.read_csv(csv_path)
    total_amount = index_file.shape[0]
    fps = 20
    for i in tqdm(range(total_amount)):
        source_path = index_file.loc[i]['source_path']
        new_name = index_file.loc[i]['new_name']
        data = np.load(source_path)
        start_frame = index_file.loc[i]['start_frame']
        end_frame = index_file.loc[i]['end_frame']
        if 'humanact12' not in source_path:
            if 'Eyes_Japan_Dataset' in source_path:
                data = data[3*fps:]
            if 'MPI_HDM05' in source_path:
                data = data[3*fps:]
            if 'TotalCapture' in source_path:
                data = data[1*fps:]
            if 'MPI_Limits' in source_path:
                data = data[1*fps:]
            if 'Transitions_mocap' in source_path:
                data = data[int(0.5*fps):]
            data = data[start_frame:end_frame]
            data[..., 0] *= -1
        
        data_m = swap_left_right(data)
        # save_path = pjoin(save_dir, )
        np.save(pjoin(save_dir, new_name), data)
        np.save(pjoin(save_dir, 'M'+new_name), data_m)

In [ ]:

save_custom_csv()
save_to_npz('./custom.csv')
save_to_npz('./original-index.csv')


